# Proceso de Ciencia de Datos - Análisis Exploratorio de Datos (Data Wrangling)

## Dataset: Hubway Bike Sharing (Boston 2011-2013)

**Autor:** Guerra Chura Joan Leonardo

Este notebook desarrolla un análisis exploratorio del sistema Hubway siguiendo las etapas del laboratorio: metadata, comprensión de datos, limpieza, detección de anomalías, visualización y planteamiento de problemas potenciales.


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from scipy import stats as sstats

%matplotlib inline
sns.set_theme(style="whitegrid")
pd.set_option('display.max_columns', 30)


# Carga de datos

Se utilizan las tablas:

- `hubway_stations.csv`: información de estaciones.
- `hubway_trips.csv`: registros individuales de viajes.


In [ ]:
# Ajustar la ruta según ubicación del repositorio
stations = pd.read_csv("../data/hubway_stations.csv")
trips = pd.read_csv("../data/hubway_trips.csv", low_memory=False)

print("Estaciones:", stations.shape)
print("Viajes:", trips.shape)


# Paso 0: Metadata

Cada registro representa:

- Una estación física en la tabla stations.
- Un viaje individual en la tabla trips.

La tabla de viajes permite analizar comportamiento temporal, espacial y características del usuario.


# Paso 1: Análisis del comportamiento de los datos

Se revisan:

- dimensiones,
- tipos de datos,
- duplicados,
- valores faltantes,
- granularidad temporal y espacial.


In [ ]:
trips.info()


In [ ]:
print("Duplicados trips:", trips.duplicated().sum())
print("Duplicados stations:", stations.duplicated().sum())

print("\nValores nulos:")
display(trips.isna().sum().sort_values(ascending=False))


In [ ]:
trips['start_dt'] = pd.to_datetime(trips['start_date'])
trips['end_dt'] = pd.to_datetime(trips['end_date'])

print(trips[['start_dt','end_dt']].describe())


## Análisis de valores faltantes

Los valores nulos deben interpretarse según el proceso de captura.

Se evalúa si los datos faltantes corresponden a errores o si contienen información sobre el tipo de usuario.


In [ ]:
pd.crosstab(
    trips['subsc_type'],
    trips['gender'].isna(),
    normalize='index'
)


# Paso 2: Análisis de outliers

La variable principal analizada será `duration`.

Se separan:

- errores de calidad (valores imposibles),
- eventos operativos reales (viajes extremadamente largos).


In [ ]:
q1 = trips['duration'].quantile(0.25)
q3 = trips['duration'].quantile(0.75)

iqr = q3-q1
upper = q3 + 1.5*iqr

print("Q1:", q1)
print("Q3:", q3)
print("Límite superior:", upper)

print("Duraciones negativas:", (trips.duration < 0).sum())
print("Mayores a 24 horas:", (trips.duration > 86400).sum())


# Paso 3: Visualización

Se analizan patrones:

- tipo de usuario,
- duración,
- horario,
- estacionalidad,
- estaciones con mayor demanda.


In [ ]:
trips['subsc_type'].value_counts().plot(kind='bar',
                                              figsize=(6,4),
                                              title='Viajes por tipo de usuario')
plt.ylabel('Cantidad de viajes')
plt.show()


In [ ]:
plt.figure(figsize=(7,4))
sns.histplot(trips[trips.duration>0]['duration'], bins=50)
plt.title("Distribución de duración de viajes")
plt.xlabel("Segundos")
plt.show()


In [ ]:
trips['hour'] = trips['start_dt'].dt.hour

plt.figure(figsize=(8,4))
trips['hour'].value_counts().sort_index().plot(kind='bar')
plt.title("Viajes por hora del día")
plt.xlabel("Hora")
plt.ylabel("Viajes")
plt.show()


In [ ]:
top_start = trips['strt_statn'].value_counts().head(10)

plt.figure(figsize=(8,4))
top_start.plot(kind='bar')
plt.title("Top 10 estaciones con más salidas")
plt.xlabel("Estación")
plt.ylabel("Número de viajes")
plt.show()


# Paso 4: Problema potencial encontrado

## Influencia del clima y distribución espacial sobre la demanda del sistema Hubway

### Pregunta de investigación

¿Cómo afectan las condiciones meteorológicas y la ubicación geográfica de las estaciones al número de viajes diarios?

### Datos involucrados

Dataset Hubway:

- fecha,
- estación origen,
- estación destino,
- coordenadas,
- tipo de usuario.

Dataset externo:

- temperatura,
- precipitación,
- nieve.

### Hipótesis

1. La lluvia reduce la demanda diaria.
2. La temperatura influye en la cantidad de viajes.
3. Los usuarios ocasionales son más sensibles al clima que los usuarios registrados.

Este planteamiento integra análisis temporal y espacial.


# Conclusión

El análisis exploratorio permite identificar:

- problemas de calidad en variables,
- valores extremos,
- patrones horarios y estacionales,
- diferencias entre usuarios,
- posibles relaciones entre ambiente y demanda.

Los resultados sirven como base para futuros modelos predictivos.
